# Face Mask Detection using CNN
This notebook focuses on building a Convolutional Neural Network to detect whether a person is wearing a face mask or not.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

## Data Preparation
Assuming the dataset is organized in `data/with_mask` and `data/without_mask`.

In [ ]:
with_mask_path = 'data/with_mask/'
without_mask_path = 'data/without_mask/'

if os.path.exists(with_mask_path) and os.path.exists(without_mask_path):
    with_mask_files = os.listdir(with_mask_path)
    without_mask_files = os.listdir(without_mask_path)
    print(f"With mask: {len(with_mask_files)}")
    print(f"Without mask: {len(without_mask_files)}")
    
    # Create labels
    with_mask_labels = [1] * len(with_mask_files)
    without_mask_labels = [0] * len(without_mask_files)
    labels = with_mask_labels + without_mask_labels
    
    # Process images
    data = []
    for img_file in with_mask_files:
        image = Image.open(with_mask_path + img_file)
        image = image.resize((128, 128))
        image = image.convert('RGB')
        image = np.array(image)
        data.append(image)
        
    for img_file in without_mask_files:
        image = Image.open(without_mask_path + img_file)
        image = image.resize((128, 128))
        image = image.convert('RGB')
        image = np.array(image)
        data.append(image)
        
    X = np.array(data)
    Y = np.array(labels)
    
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
    X_train_scaled = X_train / 255.0
    X_test_scaled = X_test / 255.0
else:
    print("Dataset directories not found. Please ensure 'data/with_mask' and 'data/without_mask' exist.")

## Build and Train CNN Model

In [ ]:
model = keras.Sequential([
    keras.layers.Conv2D(32, kernel_size=(3,3), activation='relu', input_shape=(128,128,3)),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.Conv2D(64, kernel_size=(3,3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['acc'])

if 'X_train_scaled' in locals():
    history = model.fit(X_train_scaled, Y_train, validation_split=0.1, epochs=5)

## Evaluation

In [ ]:
if 'X_test_scaled' in locals():
    loss, accuracy = model.evaluate(X_test_scaled, Y_test)
    print(f'Test Accuracy: {accuracy}')
    
    # Visualizing prediction
    sample_img = X_test[0]
    plt.imshow(sample_img)
    plt.show()
    
    prediction = model.predict(np.expand_dims(X_test_scaled[0], axis=0))
    result = np.argmax(prediction)
    print("Prediction:", "Mask" if result == 1 else "No Mask")